# 01 - Environment & Dataset Intake

Dataset: [`animeshmahajan/thermal-image-dataset`](https://www.kaggle.com/datasets/animeshmahajan/thermal-image-dataset)

Kernel: **thermal_env**

## Phần A - Environment Setup

Kiểm tra GPU NVIDIA, PyTorch có nhận CUDA không, và các thư viện cần thiết.

In [1]:
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

Fri Jul 17 15:23:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 596.36                 Driver Version: 596.36         CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3060 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   49C    P8              9W /   95W |     147MiB /   6144MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
    x = torch.rand(3, 3).cuda()
    print(x @ x)

torch: 2.11.0+cu128
cuda available: True
device: NVIDIA GeForce RTX 3060 Laptop GPU


tensor([[1.3410, 1.0996, 0.6528],
        [1.3840, 1.0746, 0.6734],
        [0.6064, 0.4812, 0.2625]], device='cuda:0')


In [3]:
import cv2
import PIL
import numpy as np
import pandas as pd
import kagglehub

print("opencv:", cv2.__version__)
print("pillow:", PIL.__version__)
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("kagglehub:", kagglehub.__version__)

opencv: 5.0.0
pillow: 12.2.0
numpy: 2.2.6
pandas: 2.3.3
kagglehub: 1.0.2


## Phần B - Dataset Intake

Tải dataset và khảo sát toàn diện: format ảnh, video/FPS, metadata, annotation, tính toàn vẹn.

### B1. Download dataset

Kiểm tra cache trước, chỉ gọi `kagglehub` khi chưa tải về để tránh tải lại nhiều lần.

In [4]:
import os

DATASET_SLUG = "animeshmahajan/thermal-image-dataset"

# kagglehub already caches under ~/.cache/kagglehub, but we check first
# so re-running this cell doesn't even hit the network when already downloaded.
cache_dir = os.path.join(
    os.path.expanduser("~"), ".cache", "kagglehub", "datasets",
    *DATASET_SLUG.split("/"), "versions"
)
cached_versions = os.path.isdir(cache_dir) and os.listdir(cache_dir)

if cached_versions:
    path = os.path.join(cache_dir, sorted(cached_versions)[-1])
    print("Dataset already downloaded, using cache:", path)
else:
    path = kagglehub.dataset_download(DATASET_SLUG)
    print("Path to dataset files:", path)

Dataset already downloaded, using cache: C:\Users\hotha\.cache\kagglehub\datasets\animeshmahajan\thermal-image-dataset\versions\1


### B2. Quét toàn bộ file

Liệt kê tất cả file trong dataset và phân loại theo đuôi: ảnh, video, annotation (YOLO/COCO/VOC).

In [ ]:
import json
import hashlib
import xml.etree.ElementTree as ET
from collections import Counter, defaultdict

from PIL import Image

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv", ".wmv"}
META_KEYWORDS = ("camera", "fov", "distance", "temperature", "weather",
                 "timestamp", "condition", "sensor", "lens")

root = path  # `path` từ cell tải dataset ở trên

all_files = []
for dirpath, _, filenames in os.walk(root):
    for fn in filenames:
        all_files.append(os.path.join(dirpath, fn))

images = [f for f in all_files if os.path.splitext(f)[1].lower() in IMAGE_EXTS]
videos = [f for f in all_files if os.path.splitext(f)[1].lower() in VIDEO_EXTS]
yolo_txts = [f for f in all_files if f.lower().endswith(".txt")
             and os.path.basename(f).lower() not in ("classes.txt", "notes.txt")]
coco_jsons = [f for f in all_files if f.lower().endswith(".json")]
voc_xmls = [f for f in all_files if f.lower().endswith(".xml")]

print(f"Tổng số file: {len(all_files)}")
print(f"  Ảnh: {len(images)} | Video: {len(videos)}")
print(f"  YOLO .txt: {len(yolo_txts)} | COCO .json: {len(coco_jsons)} | VOC .xml: {len(voc_xmls)}")

### B3. Format & độ phân giải ảnh

Verify từng ảnh mở được không, đối chiếu đuôi file với format thực tế, thống kê độ phân giải / color mode, và tính MD5 hash để phục vụ bước kiểm tra trùng lặp sau.

In [ ]:
res_counter = Counter()
mode_counter = Counter()
fmt_mismatch = []
corrupted = []
zero_byte = []
hash_map = defaultdict(list)

for f in images:
    if os.path.getsize(f) == 0:
        zero_byte.append(f)
        continue
    try:
        with Image.open(f) as im:
            im.verify()
        with Image.open(f) as im:
            res_counter[im.size] += 1
            mode_counter[im.mode] += 1
            ext = os.path.splitext(f)[1].lower().lstrip(".")
            real_fmt = (im.format or "").lower()
            if ext in ("jpg", "jpeg") and real_fmt != "jpeg":
                fmt_mismatch.append((f, real_fmt))
            elif ext == "png" and real_fmt != "png":
                fmt_mismatch.append((f, real_fmt))
        with open(f, "rb") as fh:
            h = hashlib.md5(fh.read()).hexdigest()
        hash_map[h].append(f)
    except Exception as e:
        corrupted.append((f, str(e)))

print("Độ phân giải (top 10 phổ biến nhất):")
for size, cnt in res_counter.most_common(10):
    print(f"  {size[0]}x{size[1]}: {cnt} ảnh")
print(f"Số độ phân giải khác nhau: {len(res_counter)}")
print(f"Color mode: {dict(mode_counter)}")
if fmt_mismatch:
    print(f"CẢNH BÁO: {len(fmt_mismatch)} file có đuôi không khớp format thực tế, ví dụ: {fmt_mismatch[:3]}")
else:
    print("Đuôi file khớp với format thực tế (không có mismatch).")

### B4. Video (FPS / số frame / độ phân giải)

Nếu dataset có video, đọc thông số bằng OpenCV. Dataset này hiện là ảnh tĩnh nên phần này sẽ báo không có video.

In [ ]:
if not videos:
    print("Không tìm thấy file video nào trong dataset -> đây là dataset ẢNH tĩnh, không áp dụng FPS.")
else:
    for f in videos:
        cap = cv2.VideoCapture(f)
        fps = cap.get(cv2.CAP_PROP_FPS)
        n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        print(f"  {f}: {w}x{h}, {fps:.2f} FPS, {n_frames} frames")
        cap.release()

### B5. Metadata thu thập

Kiểm tra EXIF (camera model, timestamp, GPS...) và tìm file phụ trợ có thể chứa thông tin FOV / khoảng cách lắp đặt / điều kiện môi trường.

In [ ]:
sample_n = min(30, len(images))
exif_found = 0
for f in images[:sample_n]:
    with Image.open(f) as im:
        exif = im.getexif()
        if exif and len(exif) > 0:
            exif_found += 1

print(f"EXIF: {exif_found}/{sample_n} ảnh mẫu có EXIF data.")
if exif_found == 0:
    print("-> Ảnh KHÔNG có EXIF: không có camera model, FOV, timestamp, GPS trong file ảnh.")

meta_candidate_files = [f for f in all_files
                         if os.path.splitext(f)[1].lower() in (".csv", ".json", ".yaml", ".yml", ".txt")
                         and any(k in f.lower() for k in META_KEYWORDS)]
if meta_candidate_files:
    print(f"Tìm thấy {len(meta_candidate_files)} file có tên liên quan metadata: {meta_candidate_files[:10]}")
else:
    print("Không tìm thấy file metadata riêng (camera/FOV/khoảng cách/nhiệt độ/thời tiết/ngày-đêm/timestamp).")
    print("=> Dataset này KHÔNG cung cấp thông tin thu thập (camera model, FOV, điều kiện môi trường, timestamp).")
    print("   Nếu cần các trường này, phải lấy từ nguồn khác hoặc tài liệu (documentation) của tác giả dataset.")

### B6. Annotation format & danh sách class

Đối chiếu 3 định dạng annotation có trong dataset: YOLO (`.txt`), COCO (`.json`), VOC (`.xml`).

*Lưu ý: cell dưới đây quét GỘP CẢ 3 nguồn txt (root + `All_In_One_Anot_Yolo` + `ALL_IN_ONE_RGB_ANOT_YOLO`) để khảo sát toàn cảnh. Quyết định chính thức dùng bộ **All_In_One** làm nguồn annotation (không dùng RGB) được verify riêng ở cell B8 bên dưới và ghi trong `configs/dataset.yaml`.*

In [ ]:
class_id_counter = Counter()
yolo_class_ids = set()
for f in yolo_txts:
    try:
        with open(f, "r") as fh:
            for line in fh:
                line = line.strip()
                if not line:
                    continue
                cid = line.split()[0]
                yolo_class_ids.add(cid)
                class_id_counter[cid] += 1
    except Exception:
        pass

classes_file = [f for f in all_files if os.path.basename(f).lower() in ("classes.txt", "data.yaml", "notes.json")]
print(f"[YOLO] {len(yolo_txts)} file .txt | class id xuất hiện: {sorted(yolo_class_ids)}")
print(f"       số lượng bbox / class id: {dict(class_id_counter)}")
if classes_file:
    print(f"       File ánh xạ tên class: {classes_file}")
else:
    print("       Không tìm thấy classes.txt/data.yaml -> chỉ biết class ID, chưa chắc tên class thật.")

coco_categories = {}
for f in coco_jsons:
    with open(f) as fh:
        d = json.load(fh)
    cats = {c["id"]: c["name"] for c in d.get("categories", [])}
    coco_categories[f] = {
        "num_images": len(d.get("images", [])),
        "num_annotations": len(d.get("annotations", [])),
        "categories": cats,
    }
    print(f"[COCO] {f}\n       images={coco_categories[f]['num_images']} "
          f"annotations={coco_categories[f]['num_annotations']} categories={cats}")

voc_classes = Counter()
for f in voc_xmls:
    try:
        tree = ET.parse(f)
        for obj in tree.findall("object"):
            name = obj.find("name")
            if name is not None:
                voc_classes[name.text] += 1
    except Exception:
        pass
print(f"[VOC]  {len(voc_xmls)} file .xml | class distribution: {dict(voc_classes)}")

### B7. Tính toàn vẹn dữ liệu

Kiểm tra file hỏng / rỗng, ảnh trùng lặp (MD5), và cặp ảnh-annotation bị thiếu (**đối chiếu với txt ở root**, chỉ để khảo sát toàn cảnh - xem B8 cho con số chính xác theo bộ All_In_One đã chọn).

In [ ]:
print(f"File 0 byte: {len(zero_byte)}" + (f" -> {zero_byte[:5]}" if zero_byte else ""))
print(f"Ảnh hỏng (không mở được): {len(corrupted)}" + (f" -> {corrupted[:5]}" if corrupted else ""))

duplicates = {h: fs for h, fs in hash_map.items() if len(fs) > 1}
n_dup_files = sum(len(fs) - 1 for fs in duplicates.values())
print(f"Ảnh trùng lặp (cùng MD5): {len(duplicates)} nhóm, dư ra {n_dup_files} file")
if duplicates:
    for h, fs in list(duplicates.items())[:3]:
        print(f"   nhóm trùng: {fs}")

# missing pairs: jpg (root only) <-> txt (root only)
root_dir = os.path.join(root, "ALL_IN_ONE_RGB_IMG_ANOT") if os.path.isdir(
    os.path.join(root, "ALL_IN_ONE_RGB_IMG_ANOT")) else root
root_jpgs = {os.path.splitext(f)[0] for f in os.listdir(root_dir)
             if f.lower().endswith((".jpg", ".jpeg"))} if os.path.isdir(root_dir) else set()
root_txts = {os.path.splitext(f)[0] for f in os.listdir(root_dir)
             if f.lower().endswith(".txt")} if os.path.isdir(root_dir) else set()

missing_txt = root_jpgs - root_txts
missing_jpg = root_txts - root_jpgs
print(f"Ảnh không có annotation .txt tương ứng: {len(missing_txt)}"
      + (f" -> {list(missing_txt)[:5]}" if missing_txt else ""))
print(f"File .txt không có ảnh tương ứng: {len(missing_jpg)}"
      + (f" -> {list(missing_jpg)[:5]}" if missing_jpg else ""))

print("\nHOÀN TẤT KHẢO SÁT.")

### B8. Verify nguồn annotation chính thức (All_In_One)

Dataset có 2 bản export annotation ("All_In_One" và "RGB", ít ảnh hơn) song song với txt gốc ở root.
Theo quyết định trong `configs/dataset.yaml`, dự án dùng bộ **All_In_One** - kiểm tra lại chính xác số ảnh
thiếu annotation theo bộ này (khác với con số "4" ở B7 vì đó là tính theo txt root).

In [ ]:
all_in_one_dir = os.path.join(root_dir, "Anotations", "All_In_One_Anot_Yolo")
rgb_dir = os.path.join(root_dir, "Anotations", "ALL_IN_ONE_RGB_ANOT_YOLO")

all_in_one_stems = {os.path.splitext(f)[0] for f in os.listdir(all_in_one_dir)
                     if f.lower().endswith(".txt")}
rgb_stems = {os.path.splitext(f)[0] for f in os.listdir(rgb_dir)
             if f.lower().endswith(".txt")}

print(f"Số ảnh (root): {len(root_jpgs)}")
print(f"Số file annotation - All_In_One: {len(all_in_one_stems)} | RGB: {len(rgb_stems)}")
print(f"=> Dùng All_In_One vì bao phủ nhiều ảnh hơn ({len(all_in_one_stems)} > {len(rgb_stems)})")

missing_in_all_in_one = root_jpgs - all_in_one_stems
print(f"Ảnh thiếu annotation trong bộ All_In_One (nguồn chính thức): {len(missing_in_all_in_one)}")
print(f"  ví dụ: {sorted(missing_in_all_in_one)[:5]}")

# sanity check: nội dung file trùng tên giữa root và All_In_One phải giống hệt nhau
common = sorted(root_txts & all_in_one_stems)[:5]
mismatched_content = []
for stem in common:
    with open(os.path.join(root_dir, stem + ".txt")) as f1, \
         open(os.path.join(all_in_one_dir, stem + ".txt")) as f2:
        if f1.read() != f2.read():
            mismatched_content.append(stem)
print(f"Sanity check nội dung ({len(common)} file mẫu): mismatch = {mismatched_content or 'không có'}")